# T3 — Diagnóstico y mejora del procesamiento con RDD
**Semana 13 | Unidad III — Big Data | UPN 2026-2**

**Caso ficticio y datos simulados.** Una operadora de cadena de frío transporta insumos que deben mantenerse entre 2 °C y 8 °C. El tiempo real de transporte se compara con una meta en minutos. No hay información de pacientes ni de personas reales.

**Normas de la actividad:** usar RDD (no DataFrames ni Spark SQL); mantener el CSV en la misma carpeta que este notebook; ejecutar secuencialmente; explicar toda modificación en el informe.


# Datos del grupo
- Estudiante 1: 
- Estudiante 2: 
- Estudiante 3: 
- Estudiante 4: 
- Estudiante 5: 

## 1. Preparar Spark y cargar el CSV

Ejecute las celdas siguientes en orden. La prueba inicial verifica la comunicación Spark–Python antes de procesar datos. Un fallo de infraestructura no debe confundirse con un error lógico del caso.

### Comprobación previa y resolución de problemas (v1.1)

1. Abra **Anaconda Prompt**, active el entorno `etl_bi` e inicie `jupyter lab` desde allí. Seleccione el kernel correspondiente.
2. En JupyterLab elija **Kernel → Restart Kernel** antes de ejecutar las celdas por primera vez.
3. Ejecute **Preparación del entorno** y **Prueba previa de RDD** en ese orden. Solo después ejecute la lectura del CSV.
4. Si aparece **`Python worker failed to connect back`**, detenga la ejecución y revise el entorno: **no es un error de los datos de la evaluación**. El cambio a `local[2]` y la dirección local es una ayuda diagnóstica, no una solución universal.
5. Si aparece **`NameError`**, ejecute las celdas anteriores en orden desde el inicio. Si el CSV no existe, revise la carpeta desde la que se ejecuta JupyterLab.
6. Si persiste el problema, registre las versiones de Python, Java y PySpark y solicite soporte antes de continuar. No se debe desactivar el firewall como solución general.

**Alcance:** estas comprobaciones no forman parte de las actividades calificadas y no cambian los criterios ni las reglas de negocio de la T3.


In [ ]:
# PREPARACIÓN DEL ENTORNO (ejecutar tras reiniciar el kernel y antes de crear Spark)
import os
import sys
from pathlib import Path
import pyspark
from pyspark.sql import SparkSession

# El proceso Python de Spark debe utilizar el intérprete del kernel activo.
os.environ["PYSPARK_PYTHON"] = sys.executable
# En Windows, esta preferencia puede facilitar la comunicación local.
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

print("Python del kernel:", sys.executable)
print("Versión de PySpark:", pyspark.__version__)
print("Interfaz local:", os.environ["SPARK_LOCAL_IP"])

# IMPORTANTE: una sesión Spark ya iniciada puede conservar configuraciones previas.
# Si se ejecutaron antes otras celdas con Spark, reinicie el kernel primero.
spark = (SparkSession.builder
         .appName("T3-RDD-CadenaFrio")
         .master("local[2]")
         .config("spark.driver.host", "127.0.0.1")
         .config("spark.driver.bindAddress", "127.0.0.1")
         .getOrCreate())
sc = spark.sparkContext

archivo = Path("T3_Datos_Logistica_CadenaFrio.csv").resolve()
if not archivo.is_file():
    raise FileNotFoundError(
        "No se encontró T3_Datos_Logistica_CadenaFrio.csv en la carpeta de trabajo: "
        f"{Path.cwd()}. Coloque el CSV allí o ajuste la ruta 'archivo'."
    )

print("Spark:", spark.version)
print("Master:", sc.master)
print("Python Worker:", sc.pythonExec)
print("Archivo:", archivo)


In [ ]:
# PRUEBA PREVIA: verifica el funcionamiento real de un Python Worker.
# No depende de los datos de la T3 y no revela sus soluciones.
try:
    prueba = sc.parallelize([10, 20, 30], 2)
    resultado = prueba.count()
    assert resultado == 3
    print("Prueba RDD satisfactoria: 3 registros procesados.")
except Exception as exc:
    print("La prueba de RDD falló antes de leer el CSV.")
    print("Revise: 1) kernel Conda correcto; 2) Python/Java/PySpark compatibles;")
    print("3) que python.exe y java.exe puedan comunicarse por localhost;")
    print("4) mensajes de la consola que inició JupyterLab.")
    print("No modifique las reglas del caso para resolver este fallo de entorno.")
    raise


In [ ]:
import csv

def leer_linea(linea):
    # Los datos simulados no incluyen saltos de línea internos.
    return next(csv.reader([linea]))

rdd_lineas = sc.textFile(archivo.as_uri())
encabezado = rdd_lineas.first()
rdd_datos = (rdd_lineas.filter(lambda linea: linea != encabezado)
             .map(leer_linea))
columnas = leer_linea(encabezado)

def transformar(campos):
    return {
        "id": campos[0], "fecha": campos[1], "centro": campos[2],
        "ruta": campos[3], "temperatura": float(campos[4]),
        "meta": float(campos[5]),
        "real": None if campos[6] == "" else float(campos[6])
    }

registros = rdd_datos.map(transformar)
print("Filas leídas:", registros.count())
print("Ejemplo:", registros.first())


## 2. Diagnóstico de código heredado
- El notebook contiene decisiones potencialmente incorrectas.
- Identifique las que afecten la regla de negocio, describa el impacto y documente su corrección.

In [ ]:
# BLOQUE HEREDADO POR EL EQUIPO DE TI. EJECUTE Y ANALICE SU LÓGICA.
# No asuma que los resultados son correctos aunque el código ejecute.
filas_sin_repeticiones = registros.map(lambda r: tuple(sorted(r.items()))).distinct().map(dict)

# Se pretende conservar solo registros válidos.
seleccion = filas_sin_repeticiones.filter(
    lambda r: r["real"] is not None
    and r["meta"] >= 0
    and (r["temperatura"] >= 2 or r["temperatura"] <= 8)
)

# Se pretende calcular cuántas entregas llegaron tarde por centro.
pares_tardanza = seleccion.map(
    lambda r: (r["centro"], int(r["real"] >= r["meta"]))
)
resultado_inicial = pares_tardanza.reduceByKey(lambda a, b: max(a, b))
print("Resultado inicial (debe ser auditado):", sorted(resultado_inicial.collect()))


## 3. Actividades evaluadas
**A. Diagnóstico:** determine duplicados exactos, tiempos faltantes, metas no positivas y temperaturas fuera del intervalo.

**B. Corrección:** explique y corrija los problemas del bloque heredado. Una entrega es puntual cuando `real <= meta`; cuando `real > meta` existe demora.

**C. Procesamiento:** calcule por centro el número de registros válidos, puntuales, porcentaje de puntualidad y minutos acumulados de demora mediante RDD y `reduceByKey`.

**D. Validación:** contraste sus cálculos con otra operación o con una muestra manual. El informe deberá registrar la política de exclusión.

**E. Arquitectura:** compare `local[*]` con un clúster multinodo, explique driver, executors, particiones y limitaciones de su experimento. Incluya controles de integridad y privacidad.

In [ ]:
# ACTIVIDAD A. Escriba aquí su diagnóstico (código y salidas).


In [ ]:
# ACTIVIDAD B. Corrija aquí las reglas de filtrado y tardanza.


In [ ]:
# ACTIVIDADES C-D. Calcule indicadores y pruebe sus resultados.


In [ ]:
# ACTIVIDAD E. Experimento de particiones (registre condiciones y mediciones)
from time import perf_counter
# Diseñe una comparación controlada. No interprete más particiones = mayor velocidad.


## 4. Lista de control previa a la entrega
- [ ] Las celdas ejecutan en orden.
- [ ] Expliqué errores, impacto y correcciones.
- [ ] Apliqué las reglas de negocio.
- [ ] Verifiqué indicadores.
- [ ] El informe y el notebook son coherentes.
- [ ] Declaré el uso de IA, cuando corresponda.